In [1]:
import pandas as pd
import numpy as np
import joblib
import re
import unicodedata

from sklearn.metrics import accuracy_score

In [2]:
from google.colab import files

uploaded = files.upload()

Saving TFIDF_Vectorizer.pkl to TFIDF_Vectorizer.pkl
Saving Logistic_Regression_Model.pkl to Logistic_Regression_Model.pkl


In [4]:
MODEL_FILE = "Logistic_Regression_Model.pkl"

final_model = joblib.load(MODEL_FILE)

tfidf_vectorizer = joblib.load("TFIDF_Vectorizer.pkl")

print("Final model loaded:", MODEL_FILE)
print("TF-IDF vectorizer loaded")

Final model loaded: Logistic_Regression_Model.pkl
TF-IDF vectorizer loaded


In [5]:
def preprocess_text(text):
    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Remove extra spaces and line breaks
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing spaces
    text = text.strip()

    return text

In [6]:
def predict_news(news_text):

    cleaned_text = preprocess_text(news_text)

    # Convert text into TF-IDF features
    text_vector = tfidf_vectorizer.transform([cleaned_text])

    # Predict class
    prediction = final_model.predict(text_vector)[0]

    # Confidence, when supported by model
    confidence = None

    if hasattr(final_model, "predict_proba"):
        probabilities = final_model.predict_proba(text_vector)[0]
        confidence = float(np.max(probabilities)) * 100

    return prediction, confidence

In [8]:
news_text = input("Enter Telugu agricultural news: ")

prediction, confidence = predict_news(news_text)

print("\nPrediction Result")
print("--------------------------")
print("News:", news_text)
print("Predicted Class:", prediction)

if confidence is not None:
    print("Model Confidence:", round(confidence, 2), "%")
else:
    print("Model Confidence: Not directly available")

Enter Telugu agricultural news: వ్యవసాయ సమాచారం: ఈ ఎరువు అన్ని నేల సమస్యలను ఒక్క రోజులో పరిష్కరిస్తుంది.

Prediction Result
--------------------------
News: వ్యవసాయ సమాచారం: ఈ ఎరువు అన్ని నేల సమస్యలను ఒక్క రోజులో పరిష్కరిస్తుంది.
Predicted Class: MISINFORMATION
Model Confidence: 97.02 %


In [9]:
def run_prediction():

    news_text = input("\nEnter Telugu agricultural news: ")

    prediction, confidence = predict_news(news_text)

    print("\n========== PREDICTION ==========")
    print("Input News:", news_text)
    print("Predicted Class:", prediction)

    if confidence is not None:
        print("Confidence:", round(confidence, 2), "%")

    print("==============================")

    return news_text, prediction, confidence

In [10]:
news_text, prediction, confidence = run_prediction()


Enter Telugu agricultural news: వ్యవసాయ సమాచారం: ఈ ఎరువు అన్ని నేల సమస్యలను ఒక్క రోజులో పరిష్కరిస్తుంది.

========== PREDICTION ==========
Input News: వ్యవసాయ సమాచారం: ఈ ఎరువు అన్ని నేల సమస్యలను ఒక్క రోజులో పరిష్కరిస్తుంది.
Predicted Class: MISINFORMATION
Confidence: 97.02 %


In [11]:
test_news = [
    "రైతులకు ప్రభుత్వం కొత్త పథకాన్ని ప్రకటించింది",
    "ఈ ఎరువును వాడితే ఒక్క రోజులో పంట దిగుబడి రెట్టింపు అవుతుంది",
    "వ్యవసాయ శాఖ రైతులకు శిక్షణ కార్యక్రమం నిర్వహించింది"
]

for news in test_news:

    prediction, confidence = predict_news(news)

    print("\nNews:", news)
    print("Prediction:", prediction)

    if confidence is not None:
        print("Confidence:", round(confidence, 2), "%")


News: రైతులకు ప్రభుత్వం కొత్త పథకాన్ని ప్రకటించింది
Prediction: MISINFORMATION
Confidence: 74.82 %

News: ఈ ఎరువును వాడితే ఒక్క రోజులో పంట దిగుబడి రెట్టింపు అవుతుంది
Prediction: MISINFORMATION
Confidence: 97.22 %

News: వ్యవసాయ శాఖ రైతులకు శిక్షణ కార్యక్రమం నిర్వహించింది
Prediction: MISINFORMATION
Confidence: 54.79 %


In [12]:
prediction_records = []

def record_prediction(news_text, actual_label=None):

    prediction, confidence = predict_news(news_text)

    correct = (
        actual_label == prediction
        if actual_label is not None
        else None
    )

    prediction_records.append({
        "News_Text": news_text,
        "Actual_Label": actual_label,
        "Predicted_Label": prediction,
        "Confidence_Percentage": confidence,
        "Correct_Prediction": correct
    })

    return prediction, confidence

In [13]:
record_prediction(
    "రైతులకు ప్రభుత్వం కొత్త పథకాన్ని ప్రకటించింది",
    "INFORMATION"
)

record_prediction(
    "ఈ ఎరువును వాడితే ఒక్క రోజులో పంట దిగుబడి రెట్టింపు అవుతుంది",
    "MISINFORMATION"
)

record_prediction(
    "వ్యవసాయ శాఖ రైతులకు శిక్షణ కార్యక్రమం నిర్వహించింది",
    "INFORMATION"
)

prediction_df = pd.DataFrame(prediction_records)

display(prediction_df)

,News_Text,Actual_Label,Predicted_Label,Confidence_Percentage,Correct_Prediction
0,రైతులకు ప్రభుత్వం కొత్త పథకాన్ని ప్రకటించింది,INFORMATION,MISINFORMATION,74.821808,False
1,ఈ ఎరువును వాడితే ఒక్క రోజులో పంట దిగుబడి రెట్ట...,MISINFORMATION,MISINFORMATION,97.220574,True
2,వ్యవసాయ శాఖ రైతులకు శిక్షణ కార్యక్రమం నిర్వహిం...,INFORMATION,MISINFORMATION,54.789424,False


In [14]:
prediction_df.to_csv(
    "Prediction_Results.csv",
    index=False
)

prediction_df.to_excel(
    "Prediction_Testing_Dataset.xlsx",
    index=False
)

print("Prediction results saved successfully")

Prediction results saved successfully


In [15]:
from google.colab import files

files.download("Prediction_Results.csv")
files.download("Prediction_Testing_Dataset.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>